<a href="https://colab.research.google.com/github/Somilgupta07/text-summarizer-NLP/blob/main/notebooks/text_summarizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets sentencepiece accelerate -q


In [ ]:
import torch
import pandas as pd
import re
from torch.utils.data import Dataset
from transformers import(
    T5Tokenizer,
    T5ForConditionalGeneration,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq
)

In [ ]:
# Load data
train_data=pd.read_csv("samsum-test.csv")
val_data=pd.read_csv("samsum-validation.csv")

In [ ]:
# fast training subset
train_n = min(4000, len(train_data))
val_n = min(500, len(val_data))
train_data=train_data.sample(n=train_n,random_state=42).reset_index(drop=True)
val_data=val_data.sample(n=val_n,random_state=42).reset_index(drop=True)

In [ ]:
# clean text
def clean_data(text):
    text=re.sub(r'\r\n',' ',str(text))
    text=re.sub(r'\s+','',text)
    text = re.sub(r"<.*?>", " ", text)
    text=text.strip().lower()
    return text

train_data["dialogue"]=train_data["dialogue"].apply(clean_data)
train_data["summary"]=train_data["summary"].apply(clean_data)
val_data["dialogue"]=val_data["dialogue"].apply(clean_data)
val_data["summary"]=val_data["summary"].apply(clean_data)

In [ ]:
# load tokenizer
tokenizer=T5Tokenizer.from_pretrained("t5-small")


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# custom dataset
class SummarizationDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_input_len=512, max_target_len=128):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        dialogue = "summarize: " + self.data.iloc[index]['dialogue']
        summary = self.data.iloc[index]['summary']

        inputs = self.tokenizer(
            dialogue,
            max_length=self.max_input_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        targets = self.tokenizer(
            summary,
            max_length=self.max_target_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        labels = targets["input_ids"].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids": inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "labels": labels
        }

train_dataset = SummarizationDataset(train_data, tokenizer)
val_dataset = SummarizationDataset(val_data, tokenizer)

In [ ]:
# load model
model=T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [ ]:
# device
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("Using device:",device)

Using device: cuda


In [ ]:
# data collator
data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer,model=model)

In [ ]:
# training arguments
training_args=TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    # evaluation_strategy="epoch", # Temporarily removed due to TypeError
    # save_strategy="epoch",       # Temporarily removed due to TypeError
    logging_steps=100,
    save_total_limit=2,
    weight_decay=0.01,
    warmup_steps=200,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

In [ ]:
# trainer
trainer = Trainer(
    model=model,
    # tokenizer=tokenizer, # Removed as Trainer.__init__() does not accept this argument directly
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:
# train
trainer.train()

/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)


Step,Training Loss
100,4.223752
200,3.429017
300,3.084851


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=309, training_loss=3.5662718899427492, metrics={'train_runtime': 72.2566, 'train_samples_per_second': 34.004, 'train_steps_per_second': 4.276, 'total_flos': 332534806216704.0, 'train_loss': 3.5662718899427492, 'epoch': 3.0})

In [ ]:
# save model
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [ ]:
# inference function
def summarize_dialogue(dialogue):
    model.eval()
    dialogue="summarize"+clean_data(dialogue)

    inputs=tokenizer(
        dialogue,
        max_length=512,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        output_ids=model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=128,
            num_beams=4,
            early_stopping=True
        )
    summary=tokenizer.decode(output_ids[0],skip_special_tokens=True)
    return summary

In [ ]:
# test
test_dialogue = """
Alice: Are you free this evening?
Bob: Yes, after 6 pm.
Alice: Let's meet at the cafe near the mall.
Bob: Sure, see you there!
"""

print("Summary:", summarize_dialogue(test_dialogue))

Summary: aliceseesbobfreethiseveningafter6pm.


In [ ]:
"""I fine-tuned the pretrained T5-small transformer model from Hugging Face on the SAMSum dialogue summarization dataset.
First, I cleaned the dialogues and summaries using regex preprocessing.
Then I tokenized both input dialogues and target summaries using the T5 tokenizer.
After that, I trained the T5ForConditionalGeneration model using Hugging Face Trainer with PyTorch backend.
Finally, during inference, I pass a new dialogue, tokenize it, and use model.generate() with beam search to produce the summary."""

'I fine-tuned the pretrained T5-small transformer model from Hugging Face on the SAMSum dialogue summarization dataset.\nFirst, I cleaned the dialogues and summaries using regex preprocessing.\nThen I tokenized both input dialogues and target summaries using the T5 tokenizer.\nAfter that, I trained the T5ForConditionalGeneration model using Hugging Face Trainer with PyTorch backend.\nFinally, during inference, I pass a new dialogue, tokenize it, and use model.generate() with beam search to produce the summary.'